In [ ]:
#| default_exp cli

## Restored command-line tools

`install_nbskill` installs the nbskill skill and writes the project-local Codex MCP configuration. `nbskill_mcp_log` prints the recent MCP metrics and problems, while `nbskill_mcp_log_problems` limits the report to problems. These terminal commands work without an already-connected MCP client; `nbskill_mcp` remains the MCP server entry point.

In [ ]:
#| export
import json
import tomllib
from pathlib import Path
from fastcore.script import call_parse

from nbskill.mcp import format_mcp_log_report, mcp_log_report
from nbskill.nbskill import install_nbskill as _install_nbskill

In [ ]:
from fastcore.test import *

In [ ]:
#| hide
scripts = tomllib.loads(Path("pyproject.toml").read_text())["project"]["scripts"]
test_eq(scripts["install_nbskill"], "nbskill.cli:install_nbskill")
test_eq(scripts["nbskill_mcp_log"], "nbskill.cli:nbskill_mcp_log")
test_eq(scripts["nbskill_mcp_log_problems"], "nbskill.cli:nbskill_mcp_log_problems")

In [ ]:
#| export
@call_parse
def install_nbskill(
    target: str = "codex", # codex, claude, cursor, both, or custom with skills_dir
    skills_dir: str | None = None, # Parent skills directory for a custom target
    skill_name: str = "jupyter-notebooks", # Installed skill directory name
    overwrite: bool = True, # Update stale managed files and MCP entries
    install_hooks: bool = False, # Install nbdev pre-commit hooks in the current project
    restart_mcp: bool = True, # Print reconnect guidance after configuring the server
    cursor_workspace: str | None = None, # Cursor workspace, or global configuration when omitted
    codex_workspace: str | None = ".", # Project workspace for .codex/config.toml
    reference_roots: str = "~/projects", # Local Git repositories to register
    index_references: bool = True, # Index discovered references during installation
    integrate_aai_coding: bool = True, # Install current notebook routing into aai-coding
    aai_coding_dir: str | None = None, # Explicit aai-coding checkout, or auto-detect
):
    "Install current nbskill instructions, MCP configuration, and aai-coding routing."
    return _install_nbskill(
        target=target, skills_dir=skills_dir, skill_name=skill_name, overwrite=overwrite,
        install_hooks=install_hooks, restart_mcp=restart_mcp, cursor_workspace=cursor_workspace,
        codex_workspace=codex_workspace, reference_roots=reference_roots,
        index_references=index_references, integrate_aai_coding=integrate_aai_coding,
        aai_coding_dir=aai_coding_dir,
    )

In [ ]:
#| exporti
def _print_mcp_log(report, limit, json_output=False, problems_only=False):
    if json_output: print(json.dumps(report, indent=2, sort_keys=True))
    else: print(format_mcp_log_report(report, limit=limit, problems_only=problems_only))
    return None

### Read MCP logs

Use `nbskill_mcp_log` for the recent tool and problem summary. Use `nbskill_mcp_log_problems` when only failures and warnings matter. Both accept `--path`, `--limit`, and `--json-output`.

In [ ]:
#| export
@call_parse
def nbskill_mcp_log(
    path: str | None = None, # MCP JSONL log path; uses NBSKILL_MCP_LOG or ~/.nbskill-mcp.jsonl
    limit: int = 20, # Maximum problem and tool rows to print
    json_output: bool = False, # Print machine-readable JSON
):
    "Print concise nbskill MCP log metrics and recent problems."
    return _print_mcp_log(mcp_log_report(path=path, limit=limit), limit, json_output=json_output)

In [ ]:
#| export
@call_parse
def nbskill_mcp_log_problems(
    path: str | None = None, # MCP JSONL log path; uses NBSKILL_MCP_LOG or ~/.nbskill-mcp.jsonl
    limit: int = 20, # Maximum problem rows to print
    json_output: bool = False, # Print machine-readable JSON
):
    "Print only the problem-focused nbskill MCP log summary."
    return _print_mcp_log(mcp_log_report(path=path, limit=limit), limit, json_output=json_output, problems_only=True)

In [ ]:
import tomllib
from pathlib import Path

from fastcore.test import *